In [51]:
# 1. What is a neural network? What are the general steps required to build a nueral network?
# a nueral network is a type of predictive model that works in a way that is based off of how nuerons in the brain work.
# Nueral networks are generally created using backpropagation, which is an alogrithm that uses data to train the model using gradient descent. 
# To do this we typically start with a desired output and then use the training data to tune our model so that it produces the desired output.


In [52]:
# 2. Generally, how do you check the performance of a neural network? Why is this the case?
# Generally, we want to create a loss function that measures the sum of squared errors between predictions from our model.
# We want to make sure that the loss function is decreasing as we train our model and the smaller the sum of squared errors the more accurate our model.
# We do this as if our loss function is getting smaller we now our model is creating more and more similar results that are getting closer and closer to our desired output

In [53]:
# Data cleaning
from ucimlrepo import fetch_ucirepo 
import pandas as pd
import torch
from sklearn.metrics import ConfusionMatrixDisplay
# fetch dataset 
abalone = fetch_ucirepo(id=1) 
  
# data (as pandas dataframes) 
X = abalone.data.features 
y = abalone.data.targets 
  
# metadata 
print(abalone.metadata) 
  
# variable information 
print(abalone.variables)

abalone_df = pd.concat([X, y], axis=1)
abalone_df.head()

{'uci_id': 1, 'name': 'Abalone', 'repository_url': 'https://archive.ics.uci.edu/dataset/1/abalone', 'data_url': 'https://archive.ics.uci.edu/static/public/1/data.csv', 'abstract': 'Predict the age of abalone from physical measurements', 'area': 'Biology', 'tasks': ['Classification', 'Regression'], 'characteristics': ['Tabular'], 'num_instances': 4177, 'num_features': 8, 'feature_types': ['Categorical', 'Integer', 'Real'], 'demographics': [], 'target_col': ['Rings'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1994, 'last_updated': 'Mon Aug 28 2023', 'dataset_doi': '10.24432/C55C7W', 'creators': ['Warwick Nash', 'Tracy Sellers', 'Simon Talbot', 'Andrew Cawthorn', 'Wes Ford'], 'intro_paper': None, 'additional_info': {'summary': 'Predicting the age of abalone from physical measurements.  The age of abalone is determined by cutting the shell through the cone, staining it, and counting the number of rings through a microscope -- 

,Sex,Length,Diameter,Height,Whole_weight,Shucked_weight,Viscera_weight,Shell_weight,Rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


In [54]:
# strip whitspace
abalone_df = abalone_df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

In [55]:
# remove nulls (not neecessary)
abalone_df.isna().sum()

Sex               0
Length            0
Diameter          0
Height            0
Whole_weight      0
Shucked_weight    0
Viscera_weight    0
Shell_weight      0
Rings             0
dtype: int64

In [56]:
# convert sex to numeric with M = 2, F = 1, and I = 0
abalone_df['Sex'] = abalone_df['Sex'].apply(lambda x: 2 if x == 'M' else 1 if x == 'F' else 0)
abalone_df.head()

,Sex,Length,Diameter,Height,Whole_weight,Shucked_weight,Viscera_weight,Shell_weight,Rings
0,2,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,2,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,1,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,2,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,0,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


In [57]:
# drop the data points that have a number of rings with a value count of 2 or less
abalone_df= abalone_df.groupby('Rings').filter(lambda x: len(x) > 2)
abalone_df['Rings'].value_counts()

Rings
9     689
10    634
8     568
11    487
7     391
12    267
6     259
13    203
14    126
5     115
15    103
16     67
17     58
4      57
18     42
19     32
20     26
3      15
21     14
23      9
22      6
Name: count, dtype: int64

In [58]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
X = abalone_df.drop('Rings', axis=1).values
y = abalone_df['Rings'].values

X_train, X_test, y_train, y_test = train_test_split(X,y,
                                                    test_size=0.3,
                                                    random_state=50,
                                                    stratify=y)

# scale our data
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.fit_transform(X_test)

In [59]:
import torch.nn as nn
import torch.nn.functional as F # function has activation functions

# create tensors from the data
X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)

y_train = torch.LongTensor(y_train)
y_test = torch.LongTensor(y_test)

In [60]:
# artificial neural network
class ANN_Model(nn.Module):
    def __init__(self, input_features=8,
                hidden1=20, hidden2=20,
                out_features=29):
        super().__init__()
        """
        super is a computed indirect reference
        which means that it isolates changes and
        makes sure the children in the layer of
        multiple inheritance are calling the
        right parents
        """
        self.layer_1_connection = nn.Linear(input_features, hidden1)
        self.layer_2_connection = nn.Linear(hidden1, hidden2)
        self.out = nn.Linear(hidden2, out_features)
    
    def forward(self, x):
        """
        apply activation function
        """
        x = F.relu(self.layer_1_connection(x))
        x = F.relu(self.layer_2_connection(x))
        x = self.out(x)
        return x
    

In [61]:
torch.manual_seed(50)

#create an instance of the model
ann=ANN_Model()

In [62]:
# loss function 
loss_function = nn.CrossEntropyLoss()

# optimizer
optimizer = torch.optim.Adam(ann.parameters(), lr = 0.001) 

In [63]:
# run model through various epochs/iterations
final_loss = []
n_epochs = 500
for epoch in range(n_epochs):
    y_pred = ann.forward(X_train)
    loss = loss_function(y_pred, y_train)
    final_loss.append(loss)

    if epoch % 10 == 1:
        print(f'Epoch number: {epoch} with loss {loss}')

    optimizer.zero_grad() 
    loss.backward() 
    optimizer.step() 

Epoch number: 1 with loss 3.3260891437530518
Epoch number: 11 with loss 3.2477567195892334
Epoch number: 21 with loss 3.1533401012420654
Epoch number: 31 with loss 3.0391783714294434
Epoch number: 41 with loss 2.9069695472717285
Epoch number: 51 with loss 2.7687442302703857
Epoch number: 61 with loss 2.6444308757781982
Epoch number: 71 with loss 2.544985055923462
Epoch number: 81 with loss 2.4672446250915527
Epoch number: 91 with loss 2.4072723388671875
Epoch number: 101 with loss 2.35899019241333
Epoch number: 111 with loss 2.318774938583374
Epoch number: 121 with loss 2.2841238975524902
Epoch number: 131 with loss 2.253702402114868
Epoch number: 141 with loss 2.226205348968506
Epoch number: 151 with loss 2.200942277908325
Epoch number: 161 with loss 2.1776225566864014
Epoch number: 171 with loss 2.1560070514678955
Epoch number: 181 with loss 2.136045217514038
Epoch number: 191 with loss 2.1176328659057617
Epoch number: 201 with loss 2.100734233856201
Epoch number: 211 with loss 2.085

In [64]:
y_pred = []
with torch.no_grad(): # this helps decrease memory consumption
    for i, data in enumerate(X_test):
        prediction = ann(data)
        y_pred.append(prediction.argmax())


In [65]:
from sklearn.metrics import mean_absolute_error
mean_absolute_error(y_test, y_pred)


1.5971223021582734

In [66]:
from sklearn import tree
X = abalone_df.drop('Rings', axis=1).values
y = abalone_df['Rings'].values

X_train, X_test, y_train, y_test = train_test_split(X,y,
                                                    test_size=0.3,
                                                    random_state=50,
                                                    stratify=y)

# scale our data
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.fit_transform(X_test)

In [67]:
model = tree.DecisionTreeClassifier(max_depth=7, random_state=50)

In [68]:
model = model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [69]:
mean_absolute_error(y_test, y_pred)

1.6562749800159873

In [ ]:
# the neural network performed a little bit better as it had a lower mean absolute error. This may be because it is a slightly more sopisticated model that is better able to evalute how the variables impact the outcome